In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset=pd.read_csv("CKD.csv")

In [3]:
indep=dataset.drop(columns=['classification'])
dep=dataset['classification']
indep = pd.get_dummies(indep, dtype=int,drop_first=True)

In [4]:
dataset

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.000000,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.000000,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.000000,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.000000,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.000000,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,219.000000,...,37.000000,9800.000000,4.400000,no,no,no,yes,poor,no,yes
395,51.492308,70.000000,c,0.0,2.0,normal,normal,notpresent,notpresent,220.000000,...,27.000000,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes
396,51.492308,70.000000,c,3.0,0.0,normal,normal,notpresent,notpresent,110.000000,...,26.000000,9200.000000,3.400000,yes,yes,no,poor,poor,no,yes
397,51.492308,90.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,207.000000,...,38.868902,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

In [6]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
param_grid = {'criterion': ['gini', 'entropy'],
    'max_features': ['sqrt', 'log2'], # 'auto' deprecated
    'n_estimators': [10, 50, 100],}

In [8]:
grid = GridSearchCV(RandomForestClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro')
grid.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,estimator,RandomForestClassifier()
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_features': ['sqrt', 'log2'], 'n_estimators': [10, 50, ...]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,10


In [9]:
re=grid.cv_results_
grid_predictions = grid.predict(X_test)

In [10]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)

In [11]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)

The f1_macro value for best parameter {'criterion': 'entropy', 'max_features': 'log2', 'n_estimators': 10}: 0.9924946382275899


In [12]:
print(grid.best_params_)

{'criterion': 'entropy', 'max_features': 'log2', 'n_estimators': 10}


In [13]:
table=pd.DataFrame.from_dict(re)

In [14]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.093504,0.030007,0.021900,0.011315,gini,sqrt,10,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.941027,0.979240,1.000000,0.958978,0.979717,0.971792,0.020123,9
1,0.338558,0.072273,0.020084,0.007574,gini,sqrt,50,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.980342,0.957937,0.980113,0.959848,0.979717,0.971591,0.010388,10
2,0.677865,0.179805,0.043432,0.030719,gini,sqrt,100,"{'criterion': 'gini', 'max_features': 'sqrt', ...",1.000000,0.957937,0.960565,0.959848,1.000000,0.975670,0.019884,4
3,0.063739,0.017126,0.023451,0.012329,gini,log2,10,"{'criterion': 'gini', 'max_features': 'log2', ...",0.961039,0.979240,0.941307,0.959848,0.959848,0.960257,0.012003,12
4,0.434735,0.123813,0.033595,0.014718,gini,log2,50,"{'criterion': 'gini', 'max_features': 'log2', ...",1.000000,0.936016,0.960565,0.979717,1.000000,0.975260,0.024495,5
5,0.724655,0.098845,0.038210,0.006237,gini,log2,100,"{'criterion': 'gini', 'max_features': 'log2', ...",0.980342,0.936016,0.960565,0.959848,1.000000,0.967354,0.021536,11
6,0.111763,0.024038,0.022424,0.004725,entropy,sqrt,10,"{'criterion': 'entropy', 'max_features': 'sqrt...",1.000000,0.957937,0.960565,0.979717,1.000000,0.979644,0.018243,3
7,0.375269,0.045055,0.026177,0.007733,entropy,sqrt,50,"{'criterion': 'entropy', 'max_features': 'sqrt...",1.000000,0.936016,0.960565,0.979717,1.000000,0.975260,0.024495,5
8,0.642811,0.069378,0.032785,0.016354,entropy,sqrt,100,"{'criterion': 'entropy', 'max_features': 'sqrt...",1.000000,0.936016,0.960565,0.979717,1.000000,0.975260,0.024495,5
9,0.090379,0.015869,0.017230,0.004265,entropy,log2,10,"{'criterion': 'entropy', 'max_features': 'log2...",1.000000,0.979240,1.000000,0.959848,1.000000,0.987818,0.016131,1


In [15]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[51  0]
 [ 1 81]]


In [16]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

          no       0.98      1.00      0.99        51
         yes       1.00      0.99      0.99        82

    accuracy                           0.99       133
   macro avg       0.99      0.99      0.99       133
weighted avg       0.99      0.99      0.99       133



In [17]:
print(X_test)

[[-0.37073391 -0.50842064 -0.68384276 ...  0.51639778 -0.52223297
  -0.44519456]
 [ 0.85433499  1.03002255  0.94028379 ... -1.93649167 -0.52223297
  -0.44519456]
 [ 1.03809533 -0.50842064 -0.68384276 ...  0.51639778 -0.52223297
  -0.44519456]
 ...
 [-0.24822702  1.79924415 -0.68384276 ... -1.93649167 -0.52223297
  -0.44519456]
 [-0.03431114 -1.27764223  1.75234707 ...  0.51639778 -0.52223297
  -0.44519456]
 [-1.10577525 -1.27764223 -0.68384276 ...  0.51639778 -0.52223297
  -0.44519456]]


In [18]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])

0.9997608799617408